In [1]:
# 1) 한 번만: 압축 풀기
!tar xzf sample_data/EnglishFnt.tgz -C sample_data/

In [2]:
# ────────────────────────────────────────────────────────────────
# 0) EnglishFnt에서 D/N/E 데이터셋 자동 구축
# ────────────────────────────────────────────────────────────────
import os
from pathlib import Path
from PIL import Image
import shutil
import random

# 원본 폰트 이미지 경로
FNT_BASE = Path("sample_data/English/Fnt")
OUT_BASE = Path("dataset")
CLASS_MAP = {'Sample014': 'D', 'Sample024': 'N', 'Sample015': 'E'}

# dataset/D, dataset/N, dataset/E 폴더 생성 및 초기화
for c in ['D', 'N', 'E']:
    d = OUT_BASE / c
    if d.exists():
        shutil.rmtree(d)
    d.mkdir(parents=True, exist_ok=True)

# 클래스별로 파일 복사 (최대 2000장씩 샘플)
for sample, label in CLASS_MAP.items():
    src = FNT_BASE / sample
    dst = OUT_BASE / label
    files = list(src.glob("*.png"))
    random.shuffle(files)
    for i, f in enumerate(files[:2000]):
        img = Image.open(f).convert("L").resize((32,32))
        img.save(dst / f"{label}_{i:04d}.png")

print("EnglishFnt → dataset/D,N,E 자동 구축 완료")

# ────────────────────────────────────────────────────────────────
# 1) Keras 데이터셋 로드 및 증강
# ────────────────────────────────────────────────────────────────
import tensorflow as tf
from tensorflow.keras import layers, models

IMG_SIZE    = (32, 32)
BATCH_SIZE  = 64
EPOCHS      = 30
TFLITE_PATH = "dne_classifier.tflite"
CLASS_NAMES = ["D", "N", "E"]
VALID_SPLIT = 0.2
SEED        = 42

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    "dataset",
    labels="inferred",
    label_mode="categorical",
    class_names=CLASS_NAMES,
    color_mode="grayscale",
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE,
    validation_split=VALID_SPLIT,
    subset="training",
    seed=SEED
)
val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    "dataset",
    labels="inferred",
    label_mode="categorical",
    class_names=CLASS_NAMES,
    color_mode="grayscale",
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE,
    validation_split=VALID_SPLIT,
    subset="validation",
    seed=SEED
)

normalization = layers.Rescaling(1.0 / 255)
data_augmentation = tf.keras.Sequential([
    layers.RandomBrightness(0.2),
    layers.RandomContrast(0.3),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.1),
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomFlip("horizontal"),
])


def preprocess_train(x, y):
    x = tf.expand_dims(x, -1) if x.shape[-1] != 1 else x
    x = data_augmentation(x)
    x = normalization(x)
    return x, y

train_ds = train_ds.map(preprocess_train).prefetch(buffer_size=tf.data.AUTOTUNE)
val_ds   = val_ds.map(lambda x, y: (normalization(x), y)).prefetch(buffer_size=tf.data.AUTOTUNE)

# ────────────────────────────────────────────────────────────────
# 2) CNN 모델 정의 및 학습
# ────────────────────────────────────────────────────────────────
def build_dne_model(input_shape, num_classes):
    model = models.Sequential([
        layers.Input(shape=input_shape),

        layers.Conv2D(32, (3, 3), padding="same"),
        layers.BatchNormalization(),
        layers.Activation("relu"),
        layers.Conv2D(32, (3, 3), padding="same"),
        layers.BatchNormalization(),
        layers.Activation("relu"),
        layers.MaxPooling2D(pool_size=(2, 2)),
        layers.Dropout(0.25),

        layers.Conv2D(64, (3, 3), padding="same"),
        layers.BatchNormalization(),
        layers.Activation("relu"),
        layers.Conv2D(64, (3, 3), padding="same"),
        layers.BatchNormalization(),
        layers.Activation("relu"),
        layers.MaxPooling2D(pool_size=(2, 2)),
        layers.Dropout(0.3),

        layers.Conv2D(128, (3, 3), padding="same"),
        layers.BatchNormalization(),
        layers.Activation("relu"),
        layers.GlobalAveragePooling2D(),

        layers.Dense(128, activation="relu"),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation="softmax")
    ])
    return model

model = build_dne_model(IMG_SIZE + (1,), len(CLASS_NAMES))
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)
model.summary()

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

# ────────────────────────────────────────────────────────────────
# 3) TFLite 변환 및 저장
# ────────────────────────────────────────────────────────────────
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()
with open(TFLITE_PATH, "wb") as f:
    f.write(tflite_model)
print(f"3-class TFLite model saved at '{TFLITE_PATH}'")

EnglishFnt → dataset/D,N,E 자동 구축 완료
Found 3048 files belonging to 3 classes.
Using 2439 files for training.
Found 3048 files belonging to 3 classes.
Using 609 files for validation.


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 32, 32, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 32, 32, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 32, 32, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 32, 32, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 16, 16, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 16, 16, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 16, 16, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 16, 16, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 8, 8, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 8, 8, 128)      │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_4 (Activation)       │ (None, 8, 8, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             

 Total params: 157,027 (613.39 KB)

 Trainable params: 156,387 (610.89 KB)

 Non-trainable params: 640 (2.50 KB)

Epoch 1/30
39/39 ━━━━━━━━━━━━━━━━━━━━ 19s 211ms/step - accuracy: 0.4864 - loss: 1.0121 - val_accuracy: 0.6026 - val_loss: 1.1145
Epoch 2/30
39/39 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.7529 - loss: 0.6533 - val_accuracy: 0.3448 - val_loss: 1.1974
Epoch 3/30
39/39 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.8255 - loss: 0.4566 - val_accuracy: 0.3448 - val_loss: 1.3579
Epoch 4/30
39/39 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.8739 - loss: 0.3433 - val_accuracy: 0.3448 - val_loss: 1.1351
Epoch 5/30
39/39 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.8959 - loss: 0.2895 - val_accuracy: 0.3481 - val_loss: 1.2250
Epoch 6/30
39/39 ━━━━━━━━━━━━━━━━━━━━ 2s 42ms/step - accuracy: 0.9137 - loss: 0.2263 - val_accuracy: 0.3448 - val_loss: 2.2979
Epoch 7/30
39/39 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.9194 - loss: 0.2211 - val_accuracy: 0.3448 - val_loss: 3.5088
Epoch 8/30
39/39 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.9249 - loss: 0.2260 - val_accuracy: 0.4072 -

In [3]:
!unzip /content/row_images.zip -d table_cell/

Archive:  /content/row_images.zip
   creating: table_cell/row_images/
   creating: table_cell/row_images/images/
  inflating: table_cell/row_images/images/aug_batch1_001.png  
  inflating: table_cell/row_images/images/aug_batch1_002.png  
  inflating: table_cell/row_images/images/aug_batch1_003.png  
  inflating: table_cell/row_images/images/aug_batch1_004.png  
  inflating: table_cell/row_images/images/aug_batch1_005.png  
  inflating: table_cell/row_images/images/aug_batch1_006.png  
  inflating: table_cell/row_images/images/aug_batch1_007.png  
  inflating: table_cell/row_images/images/aug_batch1_008.png  
  inflating: table_cell/row_images/images/aug_batch1_009.png  
  inflating: table_cell/row_images/images/aug_batch1_010.png  
  inflating: table_cell/row_images/images/aug_batch1_011.png  
  inflating: table_cell/row_images/images/aug_batch1_012.png  
  inflating: table_cell/row_images/images/aug_batch1_013.png  
  inflating: table_cell/row_images/images/aug_batch1_014.png  
  inf

In [4]:
import os, random, shutil

# row_images.zip을 table_cell 안에 풀어서 생긴 경로
root = "/content/table_cell/row_images"

# 실제 폴더명
img_dir = os.path.join(root, "images")
label_dir = os.path.join(root, "labels")

# train/val 폴더 생성
train_img = os.path.join(root, "images/train")
val_img   = os.path.join(root, "images/val")
train_lab = os.path.join(root, "labels/train")
val_lab   = os.path.join(root, "labels/val")

os.makedirs(train_img, exist_ok=True)
os.makedirs(val_img, exist_ok=True)
os.makedirs(train_lab, exist_ok=True)
os.makedirs(val_lab, exist_ok=True)

# 이미지 목록 불러오기
images = [f for f in os.listdir(img_dir) if f.lower().endswith((".jpg", ".png"))]
random.shuffle(images)

# 80%/20% split
split_idx = int(len(images) * 0.8)
train_list = images[:split_idx]
val_list   = images[split_idx:]

# 이미지와 해당 라벨 파일 한 쌍 동시에 이동하는 함수
def move_pair(img_name, dst_img_dir, dst_lab_dir):
    base, _ = os.path.splitext(img_name)
    shutil.copy2(os.path.join(img_dir, img_name), dst_img_dir)
    shutil.copy2(os.path.join(label_dir, base + ".txt"), dst_lab_dir)

for im in train_list:
    move_pair(im, train_img, train_lab)

for im in val_list:
    move_pair(im, val_img, val_lab)

print(len(train_list), "train,", len(val_list), "val")


760 train, 190 val


In [5]:
%%writefile /content/table_cell/row_images/data.yaml
path: /content/table_cell/row_images

train: images/train
val: images/val

nc: 1 # 학습할 클래스 개수
names: ['cell']


Writing /content/table_cell/row_images/data.yaml


In [6]:
!pip install ultralytics


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 50.0 MB/s eta 0:00:00


In [7]:
from ultralytics import YOLO

# 작은 프리트레인 모델 가져오기
model = YOLO("yolov8n.pt")   # 너무 느리면 n / 더 정확히 하고 싶으면 s, m 로 바꿔도 됨

model.train(
    data="/content/table_cell/row_images/data.yaml",
    epochs=80,        # 50~100 사이 아무거나, 데이터 적으니까 너무 크게만 안 가면 됨
    imgsz=640, # YOLO 표준 입력 이미지 크기
    batch=8,
    patience=20,      # 개선 없으면 자동 조기 종료
    project="runs_table",
    name="cell_detector"
)


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.3.233 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/table_cell/row_images/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=80, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7a79ec17bda0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [8]:
from ultralytics import YOLO

best = YOLO("runs_table/cell_detector/weights/best.pt")

best.predict(
    source="/content/table_cell/row_images/images/val",
    conf=0.3,
    save=True
)



image 1/190 /content/table_cell/row_images/images/val/aug_batch1_003.png: 96x640 124 cells, 38.7ms
image 2/190 /content/table_cell/row_images/images/val/aug_batch1_004.png: 96x640 124 cells, 6.3ms
image 3/190 /content/table_cell/row_images/images/val/aug_batch1_009.png: 96x640 124 cells, 6.6ms
image 4/190 /content/table_cell/row_images/images/val/aug_batch1_010.png: 96x640 123 cells, 6.1ms
image 5/190 /content/table_cell/row_images/images/val/aug_batch1_013.png: 96x640 124 cells, 6.4ms
image 6/190 /content/table_cell/row_images/images/val/aug_batch1_030.png: 96x640 124 cells, 9.2ms
image 7/190 /content/table_cell/row_images/images/val/aug_batch1_031.png: 96x640 124 cells, 11.0ms
image 8/190 /content/table_cell/row_images/images/val/aug_batch1_033.png: 96x640 124 cells, 10.3ms
image 9/190 /content/table_cell/row_images/images/val/aug_batch1_041.png: 96x640 124 cells, 11.0ms
image 10/190 /content/table_cell/row_images/images/val/aug_batch1_043.png: 96x640 124 cells, 9.6ms
image 11/190 /

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'cell'}
 obb: None
 orig_img: array([[[255, 255, 255],
         [255, 255, 255],
         [255, 255, 255],
         ...,
         [255, 255, 255],
         [255, 255, 255],
         [255, 255, 255]],
 
        [[255, 255, 255],
         [255, 255, 255],
         [255, 255, 255],
         ...,
         [255, 255, 255],
         [255, 255, 255],
         [255, 255, 255]],
 
        [[255, 255, 255],
         [255, 255, 255],
         [255, 255, 255],
         ...,
         [255, 255, 255],
         [255, 255, 255],
         [255, 255, 255]],
 
        ...,
 
        [[255, 255, 255],
         [255, 255, 255],
         [255, 255, 255],
         ...,
         [255, 255, 255],
         [255, 255, 255],
         [255, 255, 255]],
 
        [[255, 255, 255],
         [255, 255, 255],
         [255, 255, 255],
         ...,
         [255, 255, 

In [9]:
import zipfile
import os
import shutil

zip_path = "/content/augmented_100_batch5.zip"   # ← 여기 확인 필수!
extract_dir = "/content/augmented_100_batch5"
target_dir = "/content/test_images"

print("zip 파일 존재 여부:", os.path.exists(zip_path))

# test_images 폴더 초기화
if os.path.exists(target_dir):
    shutil.rmtree(target_dir)
os.makedirs(target_dir, exist_ok=True)

# 압축 풀기
print("압축 해제 중...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)
print("압축 해제 완료!")
print("풀린 최상위 폴더 목록:", os.listdir(extract_dir))

# PNG/JPG 파일만 test_images로 이동
count = 0
for root, dirs, files in os.walk(extract_dir):
    for f in files:
        if f.lower().endswith((".png", ".jpg", ".jpeg")):
            shutil.copy2(os.path.join(root, f), target_dir)
            count += 1

print(f"총 {count}개의 이미지가 test_images 폴더에 복사됨.")
print("test_images 안:", os.listdir(target_dir)[:10])
print("테스트 이미지 위치:", target_dir)
print("준비 완료!")

zip 파일 존재 여부: True
압축 해제 중...
압축 해제 완료!
풀린 최상위 폴더 목록: ['aug_batch5_050.png', 'aug_batch5_020.png', 'aug_batch5_084.png', 'aug_batch5_069.png', 'aug_batch5_053.png', 'aug_batch5_074.png', 'aug_batch5_009.png', 'aug_batch5_035.png', 'aug_batch5_012.png', 'aug_batch5_029.png', 'aug_batch5_005.png', 'aug_batch5_051.png', 'aug_batch5_034.png', 'aug_batch5_065.png', 'aug_batch5_067.png', 'aug_batch5_017.png', 'aug_batch5_045.png', 'aug_batch5_066.png', 'aug_batch5_042.png', 'aug_batch5_097.png', 'aug_batch5_056.png', 'aug_batch5_025.png', 'aug_batch5_019.png', 'aug_batch5_039.png', 'aug_batch5_038.png', 'aug_batch5_100.png', 'aug_batch5_049.png', 'aug_batch5_001.png', 'aug_batch5_085.png', 'aug_batch5_071.png', 'aug_batch5_022.png', 'aug_batch5_010.png', 'aug_batch5_091.png', 'aug_batch5_046.png', 'aug_batch5_011.png', 'aug_batch5_080.png', 'aug_batch5_048.png', 'aug_batch5_094.png', 'aug_batch5_002.png', 'aug_batch5_036.png', 'aug_batch5_095.png', 'aug_batch5_008.png', 'aug_batch5_092.png',

In [18]:
import cv2
import numpy as np
from ultralytics import YOLO
import tensorflow as tf
import os
import matplotlib.pyplot as plt
import glob
import json


####################################
# 1) YOLO 모델 로드
####################################

yolo_model = YOLO("runs_table/cell_detector/weights/best.pt")


####################################
# 2) TFLite D/N/E 분류기 로드
####################################

tflite_path = "/content/dne_classifier.tflite"
if not os.path.exists(tflite_path):
    raise FileNotFoundError(f"TFLite 모델을 찾을 수 없음: {tflite_path}")

interpreter = tf.lite.Interpreter(model_path=tflite_path)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

INPUT_H = input_details[0]['shape'][1]
INPUT_W = input_details[0]['shape'][2]
INPUT_C = input_details[0]['shape'][3]

print("TFLite input shape:", (INPUT_H, INPUT_W, INPUT_C))

CLASS_NAMES = ['D', 'N', 'E']


####################################
# 3) 빈칸 판단 함수 (강화)
####################################
def is_blank_cell(crop, dark_thresh=210, min_dark_ratio=0.02):
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    dark_ratio = np.mean(gray < dark_thresh)
    return dark_ratio < min_dark_ratio


####################################
# 4) TFLite 분류
####################################
def predict_dne_tflite(crop_resized):
    x = np.expand_dims(crop_resized, axis=0)

    interpreter.set_tensor(input_details[0]['index'], x)
    interpreter.invoke()
    output = interpreter.get_tensor(output_details[0]['index'])

    probs = output[0]
    pred_idx = int(np.argmax(probs))
    return CLASS_NAMES[pred_idx], float(probs[pred_idx])


####################################
# 5) YOLO 박스 기반 분류 (1열은 일단 제거 X)
####################################
def classify_row_cells(image_path, conf=0.3):
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError(f"이미지를 열 수 없음: {image_path}")

    result = yolo_model(image_path, conf=conf)[0]

    if result.boxes is None or len(result.boxes) == 0:
        print("YOLO가 아무 셀도 못 찾음.")
        return [], []

    cells = []
    for box in result.boxes:
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
        cx = (x1 + x2) / 2
        cy = (y1 + y2) / 2

        cells.append({
            "cx": cx,
            "cy": cy,
            "bbox": (x1, y1, x2, y2)
        })

    # YOLO 박스는 일단 그대로 반환 후 행 클러스터링에서 처리함
    return cells


####################################
# 6) 이미지 위에 라벨 그리기
####################################
def draw_dne_on_row(image_path, labels_with_prob, boxes, out_path):
    img = cv2.imread(image_path)

    for ((label, prob), (x1, y1, x2, y2)) in zip(labels_with_prob, boxes):
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(img, label, (x1, max(y1 - 5, 0)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)

    cv2.imwrite(out_path, img)
    print("저장됨:", out_path)


####################################
# 7) 여러 장 테스트 자동 처리
####################################

N_ROWS = 4
INPUT_FOLDER = "/content/test_images"
SAVE_FOLDER  = "/content/ocr_result_imgs"
os.makedirs(SAVE_FOLDER, exist_ok=True)

all_results = {}

test_files = sorted(glob.glob(INPUT_FOLDER + "/*.png")) + \
             sorted(glob.glob(INPUT_FOLDER + "/*.jpg"))

print("총 이미지 개수:", len(test_files))


for img_path in test_files:
    fname = os.path.basename(img_path)
    print(f"\n===== Processing {fname} =====")

    # ---------- YOLO 박스 추출 ----------
    cell_boxes = classify_row_cells(img_path, conf=0.3)

    img = cv2.imread(img_path)
    H, W = img.shape[:2]

    # ---------- 1) 행 자동 클러스터링 ----------
    cell_boxes.sort(key=lambda c: c["cy"])

    rows = [[] for _ in range(N_ROWS)]
    current_row = 0
    CY_GAP = H * 0.10

    prev_cy = cell_boxes[0]["cy"]
    rows[current_row].append(cell_boxes[0])

    for c in cell_boxes[1:]:
        if abs(c["cy"] - prev_cy) > CY_GAP and current_row < N_ROWS - 1:
            current_row += 1
        rows[current_row].append(c)
        prev_cy = c["cy"]

    # ---------- 2) 행 정렬 후 1열 삭제 ----------
    for r in range(N_ROWS):
        rows[r].sort(key=lambda c: c["cx"])
        if len(rows[r]) > 0:
            rows[r] = rows[r][1:]   # 🔥 각 행에서 가장 왼쪽 칸 삭제

    # ---------- 3) 분류 수행 ----------
    labels_with_prob = []
    boxes = []

    for r in range(N_ROWS):
        for c in rows[r]:
            x1, y1, x2, y2 = c["bbox"]
            crop = img[y1:y2, x1:x2]

            # 빈칸이면 '-'
            if is_blank_cell(crop):
                labels_with_prob.append(('-', None))
                boxes.append((x1, y1, x2, y2))
                continue

            # Resize + Normalize
            resized = cv2.resize(crop, (INPUT_W, INPUT_H))
            if INPUT_C == 1:
                resized = cv2.cvtColor(resized, cv2.COLOR_BGR2GRAY)
                resized = resized[..., None]
            resized = resized.astype(np.float32) / 255.0

            pred_label, pred_prob = predict_dne_tflite(resized)
            labels_with_prob.append((pred_label, pred_prob))
            boxes.append((x1, y1, x2, y2))

    # ---------- 4) JSON 저장용 배열 구성 ----------
    result_rows = []
    idx = 0

    for r in range(N_ROWS):
        row_len = len(rows[r])
        row_labels = []

        for _ in range(row_len):
            row_labels.append(labels_with_prob[idx][0])
            idx += 1

        result_rows.append(row_labels)

    all_results[fname] = result_rows

    # ---------- 5) 시각화 ----------
    out_path = f"{SAVE_FOLDER}/{fname}"
    draw_dne_on_row(img_path, labels_with_prob, boxes, out_path)


# ---------- 최종 JSON 저장 ----------
with open("/content/ocr_results.json", "w", encoding="utf-8") as f:
    json.dump(all_results, f, ensure_ascii=False, indent=2)

print("\n=== 전체 테스트셋 처리 완료! ===")
print("결과 json:", "/content/ocr_results.json")
print("시각화 폴더:", SAVE_FOLDER)

TFLite input shape: (np.int32(32), np.int32(32), np.int32(1))
총 이미지 개수: 100

===== Processing aug_batch5_001.png =====

image 1/1 /content/test_images/aug_batch5_001.png: 96x640 124 cells, 9.9ms
Speed: 3.4ms preprocess, 9.9ms inference, 1.6ms postprocess per image at shape (1, 3, 96, 640)
저장됨: /content/ocr_result_imgs/aug_batch5_001.png

===== Processing aug_batch5_002.png =====

image 1/1 /content/test_images/aug_batch5_002.png: 96x640 124 cells, 8.6ms
Speed: 1.3ms preprocess, 8.6ms inference, 2.1ms postprocess per image at shape (1, 3, 96, 640)
저장됨: /content/ocr_result_imgs/aug_batch5_002.png

===== Processing aug_batch5_003.png =====

image 1/1 /content/test_images/aug_batch5_003.png: 96x640 121 cells, 9.7ms
Speed: 1.4ms preprocess, 9.7ms inference, 2.4ms postprocess per image at shape (1, 3, 96, 640)
저장됨: /content/ocr_result_imgs/aug_batch5_003.png

===== Processing aug_batch5_004.png =====

image 1/1 /content/test_images/aug_batch5_004.png: 96x640 123 cells, 10.1ms
Speed: 1.3ms pr

In [17]:
!rm -rf ocr_result_imgs